In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\Best By\AppData\Local\Temp\ipykernel_22892\2246380887.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Best By\AppData\Local\Temp\ipykernel_22892\2246380887.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Best By\AppData\Local\Temp\ipykernel_22892\2246380887.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Best By\AppData\Local\Temp\ipykernel_22892\2246380887.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_pol

## Local MCP server

In [3]:
import sys
sys.stderr = sys.__stderr__
from langchain_mcp_adapters.client import MultiServerMCPClient
client= MultiServerMCPClient(
    {
        "local_server":{
            "transport": "stdio",
            "command": "python",
            "args":["mcp_server.py"]
        }
    }
)

In [4]:
tools= await client.get_tools()

resources= await client.get_resources("local_server")

prompt= await client.get_prompt("local_server", "prompt")

prompt=prompt[0].content

In [5]:
from langchain.agents import create_agent

agent= create_agent(
    model="gpt-5-nano",
    system_prompt=prompt,
    tools=tools,
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response =  await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]}, config=config
)


In [7]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='4b91f560-164d-4785-8248-b6e7a9af145a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 276, 'total_tokens': 432, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqQjfDsG2iTYsbh3gCgAKsXhvTiGU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec302-bd9b-7b32-83d3-3c1d588e72c0-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_o8mZmQJfEX4ymFD6D4KMtLQv', 'type': 'tool_call'}], invalid_tool_calls=[], us

## Online MCP

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [9]:
agent= create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [10]:
from langchain.messages import HumanMessage
question = "What time is it in New York right now?"

response =  await agent.ainvoke(
    {"messages": [HumanMessage(content=question)]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it in New York right now?', additional_kwargs={}, response_metadata={}, id='9207dfe3-35bc-41d2-b092-4d4a933d4e5d'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 300, 'total_tokens': 391, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqQuIk323BPCuKAgwXFDN3LjNPOjj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec30c-c526-72a2-9ad9-689425944d06-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_0VNAWSfQhjKRN9fnGBZuch94', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad